In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

# Load insurance dataset
insurance = pd.read_csv('../data/insurance_fire_census_weather_raw.csv', low_memory=False)
print(f"Insurance dataset: {insurance.shape}")
print(f"Years: {sorted(insurance['Year'].unique())}")
print(f"Zip codes: {insurance['ZIP'].nunique()}")
print(f"\nEarned Premium stats:")
print(insurance['Earned Premium'].describe().round(2))

Insurance dataset: (47033, 76)
Years: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]
Zip codes: 2251

Earned Premium stats:
count       47033.00
mean       507910.02
std       1455284.37
min          -559.00
25%          2842.00
50%         44207.00
75%        264891.00
max      36270532.00
Name: Earned Premium, dtype: float64


In [2]:
# === Understand the premium prediction task ===
# We need to predict Earned Premium for 2021 using 2018-2020 data

# How many zip codes appear in all 4 years?
zip_counts = insurance.groupby('ZIP')['Year'].nunique()
print(f"Zip codes in all 4 years: {(zip_counts == 4).sum()}")
print(f"Zip codes in 3 years (2018-2020): {(zip_counts >= 3).sum()}")
print(f"Zip codes in only 1-2 years: {(zip_counts <= 2).sum()}")

# Check if there are multiple rows per zip per year (different insurance categories)
rows_per_zip_year = insurance.groupby(['ZIP', 'Year']).size()
print(f"\nRows per zip-year:")
print(rows_per_zip_year.describe().round(2))
print(f"Max rows for a single zip-year: {rows_per_zip_year.max()}")

# Look at premium trends over time
print(f"\n=== Average Earned Premium by Year ===")
for yr in [2018, 2019, 2020, 2021]:
    subset = insurance[insurance['Year'] == yr]
    # Aggregate to zip level first
    zip_premium = subset.groupby('ZIP')['Earned Premium'].sum()
    print(f"  {yr}: mean=${zip_premium.mean():,.0f}, median=${zip_premium.median():,.0f}, n={len(zip_premium)} zips")

Zip codes in all 4 years: 1906
Zip codes in 3 years (2018-2020): 1979
Zip codes in only 1-2 years: 272

Rows per zip-year:
count    8288.00
mean        5.67
std         2.95
min         1.00
25%         5.00
50%         6.00
75%         6.00
max        84.00
dtype: float64
Max rows for a single zip-year: 84

=== Average Earned Premium by Year ===
  2018: mean=$2,072,658, median=$962,946, n=2059 zips
  2019: mean=$2,230,625, median=$1,214,020, n=1993 zips
  2020: mean=$3,431,476, median=$1,653,079, n=2118 zips
  2021: mean=$3,733,440, median=$1,831,750, n=2118 zips


In [3]:
# === Build Task 2 feature matrix ===
# Strategy: for each zip code, use 2018-2020 data to predict 2021 premium

# First aggregate to zip-year level (sum premiums, claims; average scores)
agg_rules = {
    'Earned Premium': 'sum',
    'Earned Exposure': 'sum',
    'Avg Fire Risk Score': 'mean',
    'Cov A Amount Weighted Avg': 'mean',
    'Cov C Amount Weighted Avg': 'mean',
    'CAT Cov A Fire -  Incurred Losses': 'sum',
    'CAT Cov A Fire -  Number of Claims': 'sum',
    'CAT Cov A Smoke -  Incurred Losses': 'sum',
    'CAT Cov A Smoke -  Number of Claims': 'sum',
    'Non-CAT Cov A Fire -  Incurred Losses': 'sum',
    'Non-CAT Cov A Fire -  Number of Claims': 'sum',
    'Non-CAT Cov C Fire -  Incurred Losses': 'sum',
    'Non-CAT Cov C Fire -  Number of Claims': 'sum',
    'Number of High Fire Risk Exposure': 'sum',
    'Number of Very High Fire Risk Exposure': 'sum',
    'Number of Low Fire Risk Exposure': 'sum',
    'Number of Moderate Fire Risk Exposure': 'sum',
    'Number of Negligible Fire Risk Exposure': 'sum',
    'total_population': 'first',
    'median_income': 'first',
    'total_housing_units': 'first',
    'average_household_size': 'first',
    'housing_value': 'first',
    'poverty_status': 'first',
    'median_monthly_housing_costs': 'first',
    'owner_occupied_housing_units': 'first',
    'renter_occupied_housing_units': 'first',
}

zip_year = insurance.groupby(['ZIP', 'Year']).agg(agg_rules).reset_index()
print(f"Zip-year level: {zip_year.shape}")

# === Create lagged features ===
# For each zip, use prior year values as features
zip_year = zip_year.sort_values(['ZIP', 'Year'])

# Lag features: previous year's premium, claims, exposure
lag_cols = ['Earned Premium', 'Earned Exposure', 
            'CAT Cov A Fire -  Incurred Losses', 'CAT Cov A Fire -  Number of Claims',
            'Non-CAT Cov A Fire -  Incurred Losses']

for col in lag_cols:
    zip_year[f'{col}_lag1'] = zip_year.groupby('ZIP')[col].shift(1)
    zip_year[f'{col}_lag2'] = zip_year.groupby('ZIP')[col].shift(2)

# Premium growth rate from prior year
zip_year['premium_growth'] = zip_year.groupby('ZIP')['Earned Premium'].pct_change()

# Rolling average premium (2-year)
zip_year['premium_rolling_avg'] = zip_year.groupby('ZIP')['Earned Premium'].transform(
    lambda x: x.shift(1).rolling(2, min_periods=1).mean()
)

print(f"After lag features: {zip_year.shape}")
print(f"Null counts in lag features (expected for first year):")
print(f"  lag1 nulls: {zip_year['Earned Premium_lag1'].isna().sum()}")
print(f"  lag2 nulls: {zip_year['Earned Premium_lag2'].isna().sum()}")

Zip-year level: (8288, 29)
After lag features: (8288, 41)
Null counts in lag features (expected for first year):
  lag1 nulls: 2251
  lag2 nulls: 4403


In [4]:
# === Add Task 1 fire risk predictions as a feature ===
# Load our feature matrix which has fire risk scores
fire_risk = pd.read_csv('../data/feature_matrix_clean.csv')[['zip', 'Year', 'had_fire', 'prior_fire_count', 'prior_total_acres_log']]
fire_risk = fire_risk.rename(columns={'zip': 'ZIP'})
fire_risk['ZIP'] = fire_risk['ZIP'].astype(int)
fire_risk['Year'] = fire_risk['Year'].astype(int)

zip_year = zip_year.merge(fire_risk, on=['ZIP', 'Year'], how='left')
# Fill missing fire features with 0
for col in ['had_fire', 'prior_fire_count', 'prior_total_acres_log']:
    zip_year[col] = zip_year[col].fillna(0)

print(f"After adding fire risk: {zip_year.shape}")

# === Train/test split ===
# Train on 2020 (which has lag features from 2018, 2019)
# Test on 2021 (which has lag features from 2019, 2020)
# We skip 2018 and 2019 for training because they lack lag2 features

train_task2 = zip_year[zip_year['Year'] == 2020].copy()
test_task2 = zip_year[zip_year['Year'] == 2021].copy()

# Drop rows with null lag features
train_task2 = train_task2.dropna(subset=['Earned Premium_lag1'])
test_task2 = test_task2.dropna(subset=['Earned Premium_lag1'])

# Target
y_train_t2 = train_task2['Earned Premium']
y_test_t2 = test_task2['Earned Premium']

# Features (everything except ZIP, Year, and target)
exclude_cols = ['ZIP', 'Year', 'Earned Premium']
feature_cols_t2 = [col for col in train_task2.columns if col not in exclude_cols]

# Fill remaining nulls with median
for col in feature_cols_t2:
    median_val = train_task2[col].median()
    train_task2[col] = train_task2[col].fillna(median_val)
    test_task2[col] = test_task2[col].fillna(median_val)

X_train_t2 = train_task2[feature_cols_t2]
X_test_t2 = test_task2[feature_cols_t2]

print(f"\nTrain: {X_train_t2.shape}, Test: {X_test_t2.shape}")
print(f"Train premium: mean=${y_train_t2.mean():,.0f}, median=${y_test_t2.median():,.0f}")
print(f"Test premium:  mean=${y_test_t2.mean():,.0f}, median=${y_test_t2.median():,.0f}")
print(f"\nFeatures ({len(feature_cols_t2)}):")
for f in feature_cols_t2:
    print(f"  {f}")

After adding fire risk: (8288, 44)

Train: (1979, 41), Test: (2118, 41)
Train premium: mean=$3,672,476, median=$1,831,750
Test premium:  mean=$3,733,440, median=$1,831,750

Features (41):
  Earned Exposure
  Avg Fire Risk Score
  Cov A Amount Weighted Avg
  Cov C Amount Weighted Avg
  CAT Cov A Fire -  Incurred Losses
  CAT Cov A Fire -  Number of Claims
  CAT Cov A Smoke -  Incurred Losses
  CAT Cov A Smoke -  Number of Claims
  Non-CAT Cov A Fire -  Incurred Losses
  Non-CAT Cov A Fire -  Number of Claims
  Non-CAT Cov C Fire -  Incurred Losses
  Non-CAT Cov C Fire -  Number of Claims
  Number of High Fire Risk Exposure
  Number of Very High Fire Risk Exposure
  Number of Low Fire Risk Exposure
  Number of Moderate Fire Risk Exposure
  Number of Negligible Fire Risk Exposure
  total_population
  median_income
  total_housing_units
  average_household_size
  housing_value
  poverty_status
  median_monthly_housing_costs
  owner_occupied_housing_units
  renter_occupied_housing_units
  E

In [6]:
# premium_growth has inf where prior year premium was 0
X_train_t2 = X_train_t2.replace([np.inf, -np.inf], np.nan)
X_test_t2 = X_test_t2.replace([np.inf, -np.inf], np.nan)

# Fill with median
for col in X_train_t2.columns:
    median_val = X_train_t2[col].median()
    X_train_t2[col] = X_train_t2[col].fillna(median_val)
    X_test_t2[col] = X_test_t2[col].fillna(median_val)

print(f"Infinities and nulls cleaned.")
print(f"Any nulls remaining: {X_train_t2.isna().sum().sum()}, {X_test_t2.isna().sum().sum()}")
print(f"Any inf remaining: {np.isinf(X_train_t2.values).sum()}, {np.isinf(X_test_t2.values).sum()}")

Infinities and nulls cleaned.
Any nulls remaining: 0, 0
Any inf remaining: 0, 0


In [7]:
# === Evaluation helper for regression ===
def evaluate_regression(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true.clip(lower=1))) * 100
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(f"MAE:   ${mae:,.0f}")
    print(f"RMSE:  ${rmse:,.0f}")
    print(f"R2:    {r2:.4f}")
    print(f"MAPE:  {mape:.1f}%")

# === Model 1: Random Forest ===
rf_t2 = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
rf_t2.fit(X_train_t2, y_train_t2)
rf_pred_t2 = rf_t2.predict(X_test_t2)
evaluate_regression("Random Forest", y_test_t2, rf_pred_t2)

# === Model 2: XGBoost ===
xgb_t2 = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1)
xgb_t2.fit(X_train_t2, y_train_t2)
xgb_pred_t2 = xgb_t2.predict(X_test_t2)
evaluate_regression("XGBoost", y_test_t2, xgb_pred_t2)

# === Baseline: just predict last year's premium ===
naive_pred = test_task2['Earned Premium_lag1'].values
evaluate_regression("Naive (last year's premium)", y_test_t2, naive_pred)

# === Feature importance ===
importance_t2 = pd.DataFrame({
    'feature': feature_cols_t2,
    'importance': xgb_t2.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\n=== Top 10 Features (XGBoost) ===")
for _, row in importance_t2.head(10).iterrows():
    bar = '█' * int(row['importance'] * 100)
    print(f"  {row['feature']:40s} {row['importance']:.4f} {bar}")


  Random Forest
MAE:   $402,854
RMSE:  $2,001,430
R2:    0.8691
MAPE:  754.8%

  XGBoost
MAE:   $418,345
RMSE:  $2,178,936
R2:    0.8448
MAPE:  15305.1%

  Naive (last year's premium)
MAE:   $739,875
RMSE:  $2,845,409
R2:    0.7353
MAPE:  1802.3%

=== Top 10 Features (XGBoost) ===
  premium_rolling_avg                      0.6115 █████████████████████████████████████████████████████████████
  Earned Premium_lag1                      0.1514 ███████████████
  premium_growth                           0.1311 █████████████
  Non-CAT Cov A Fire -  Incurred Losses    0.0320 ███
  Earned Exposure                          0.0286 ██
  Earned Premium_lag2                      0.0166 █
  Number of Very High Fire Risk Exposure   0.0097 
  Non-CAT Cov C Fire -  Incurred Losses    0.0062 
  Non-CAT Cov C Fire -  Number of Claims   0.0019 
  prior_total_acres_log                    0.0018 
